# Notebook Day 2:Modèles et validation groupée par patient

Equipe : Bob Le Bricoleur

Victor MICHELON, Bertrand MAIGRE, Jean-Baptiste PASTEAU, Oscar DUJARDIN

## Préparation

In [19]:
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from skore import evaluate, Project, login

X_train = pd.read_csv("data/X_train.csv")
y_train = pd.read_csv("data/y_train.csv")
X_test = pd.read_csv("data/X_test.csv")

visits = X_train.merge(y_train, on="Index")
y = visits["target"]

feature_cols = [
    "sexM",
    "age_at_diagnosis",
    "age",
    "ledd",
    "time_since_intake_on",
    "time_since_intake_off",
    "on",
    "off",
]
X = visits[feature_cols]

dummy = DummyRegressor(strategy="mean")

login(mode="hub")
project = Project(name="bobathon-esilv", mode="hub", workspace="bob-le-bricoleur")

submissions_dir = Path("submissions")
submissions_dir.mkdir(exist_ok=True)

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                                  Already logged in.                                  │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

## 1. Modèle linéaire (Ridge)

Ridge a besoin de données sans trous, donc on impute avec la médiane avant.

In [20]:
ridge = make_pipeline(
    SimpleImputer(strategy="median"),
    Ridge(alpha=1.0),
)

# comparaison dummy vs ridge (split aléatoire par défaut)
report = evaluate({"dummy": dummy, "ridge": ridge}, X, y)
report.metrics.summarize()

Output()

Estimator,dummy,ridge
Metric,,
Score,-2.315282e-06,NaN
R²,-2.315282e-06,5.949091e-01
RMSE,1.647794e+01,1.048766e+01
MAE,1.348322e+01,8.435659e+00
MAPE,7.002507e+14,3.011626e+14
Fit time (s),7.100420e-04,2.107479e-02
Predict time (s),4.295801e-05,9.713750e-04


In [21]:
# put n'accepte pas les ComparisonReport, on envoie chaque modèle séparément
project.put("02_dummy", evaluate(dummy, X, y))
project.put("02_ridge_alpha1", evaluate(ridge, X, y))

Output()

Consult your report at 
]8;id=5575123;https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/estimators/89762\https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/estimators/89762]8;;\

Output()

Consult your report at 
]8;id=5575126;https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/estimators/89763\https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/estimators/89763]8;;\

In [22]:
# coefficients de Ridge (features non scalées, donc pas comparables entre elles)
ridge.fit(X, y)
pd.Series(ridge[-1].coef_, index=feature_cols)

sexM                    -0.199699
age_at_diagnosis        -0.957885
age                      1.072983
ledd                    -0.000454
time_since_intake_on     1.704143
time_since_intake_off   -0.154672
on                       0.477556
off                      0.589665
dtype: float64

## 2. Réglage de alpha et soumission Kaggle

In [23]:
# on teste plusieurs valeurs de alpha
for alpha in [0.1, 1.0, 10.0, 100.0]:
    model = make_pipeline(SimpleImputer(strategy="median"), Ridge(alpha=alpha))
    r = evaluate(model, X, y)
    print(alpha, r.metrics.rmse())

0.1 10.487659120429758
1.0 10.487659373119959
10.0 10.487661914813827
100.0 10.487688798604573


In [24]:
ridge = make_pipeline(
    SimpleImputer(strategy="median"),
    Ridge(alpha=10.0),
)
report = evaluate(ridge, X, y)
project.put("02_ridge", report)

Output()

Consult your report at 
]8;id=5575129;https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/estimators/89764\https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/estimators/89764]8;;\

In [25]:
# on entraîne sur tout le train et on prédit le test
final = clone(ridge).fit(X, y)

submission = X_test[["Index"]].copy()
submission["target"] = final.predict(X_test[feature_cols])
submission.to_csv(submissions_dir / "02_ridge.csv", index=False)
submission.head()

,Index,target
0,66,29.902082
1,67,19.057262
2,68,37.911292
3,69,42.399635
4,70,31.591102


## 3. Validation croisée groupée par patient

Avec un split aléatoire, un même patient peut se retrouver dans le train et dans le test, ce qui donne des scores trop optimistes. Avec `GroupKFold`, un patient est soit dans le train soit dans le test.

In [26]:
groups = visits["patient_id"]
cv_splits = list(GroupKFold(n_splits=5).split(X, y, groups=groups))

report = evaluate({"dummy": dummy, "ridge": ridge}, X, y, splitter=cv_splits)
report.metrics.summarize()

Output()

Output()

Output()

mean_dummy    mean_ridge     std_dummy     std_ridge
metric                                                              
score        -8.604797e-04           NaN  5.614494e-04           NaN
r2           -8.604797e-04  5.869953e-01  5.614494e-04  1.552435e-02
rmse          1.649918e+01  1.059652e+01  3.428937e-01  2.669708e-01
mae           1.348897e+01  8.526112e+00  3.206494e-01  1.781821e-01
mape          6.549411e+14  2.843776e+14  2.115478e+14  9.894256e+13
fit_time      3.254334e-04  1.428012e-02  1.529408e-04  4.584164e-03
predict_time  5.466680e-05  9.591918e-04  3.276383e-05  8.086496e-04
Use .frame() to control the format of the output.

In [27]:
# put n'accepte pas les ComparisonReport, on envoie chaque modèle séparément
project.put("03_cv_dummy", evaluate(dummy, X, y, splitter=cv_splits))
project.put("03_cv_ridge", evaluate(ridge, X, y, splitter=cv_splits))

Output()

Output()

Consult your report at 
]8;id=5575132;https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/cross-validations/89771\https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/cross-validations/89771]8;;\

Output()

Output()

Consult your report at 
]8;id=5575135;https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/cross-validations/89777\https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/cross-validations/89777]8;;\

## 4. HistGradientBoosting

Ce modèle gère directement les valeurs manquantes, donc on ne les impute pas : le fait qu'un `off` soit absent peut être une information utile.

In [28]:
hgbr = HistGradientBoostingRegressor(random_state=0)
report = evaluate({"ridge": ridge, "hgbr": hgbr}, X, y, splitter=cv_splits)
report.metrics.summarize()

Output()

Output()

Output()

mean_ridge     mean_hgbr     std_ridge      std_hgbr
metric                                                              
r2            5.869953e-01  7.788112e-01  1.552435e-02  4.408365e-03
rmse          1.059652e+01  7.755578e+00  2.669708e-01  1.528059e-01
mae           8.526112e+00  6.104824e+00  1.781821e-01  1.026096e-01
mape          2.843776e+14  9.265305e+13  9.894256e+13  3.144672e+13
fit_time      1.412614e-02  4.465327e-01  4.370447e-03  1.532029e-02
predict_time  6.899252e-04  7.958875e-03  1.298134e-04  5.688704e-04
Use .frame() to control the format of the output.

In [29]:
project.put("04_hgbr", evaluate(hgbr, X, y, splitter=cv_splits))

Output()

Output()

Consult your report at 
]8;id=5575138;https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/cross-validations/89783\https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/cross-validations/89783]8;;\

## 5. skrub `tabular_pipeline`

Ici on garde aussi les colonnes texte (`cohort`, `gene`...). On enlève juste les identifiants et la target. Le `TableVectorizer` choisit tout seul l'encodage de chaque colonne.

In [30]:
from skrub import tabular_pipeline

X_full = visits.drop(columns=["Index", "patient_id", "target"])
model = tabular_pipeline("regressor")

report = evaluate(model, X_full, y, splitter=cv_splits)
report.metrics.summarize()

Output()

Output()

histgradientboostingregressor_mean  \
metric                                             
r2                                  7.915436e-01   
rmse                                7.528963e+00   
mae                                 5.931201e+00   
mape                                9.943257e+13   
fit_time                            6.993446e-01   
predict_time                        4.753642e-02   

              histgradientboostingregressor_std  
metric                                           
r2                                 4.679769e-03  
rmse                               1.526670e-01  
mae                                1.062154e-01  
mape                               3.664381e+13  
fit_time                           1.603529e-02  
predict_time                       1.551142e-03  
Use .frame() to control the format of the output.

In [31]:
project.put("05_tabular_pipeline", report)

Output()

Consult your report at 
]8;id=5575141;https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/cross-validations/89789\https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/cross-validations/89789]8;;\

## 6. skrub DataOps

Même modèle qu'avant, mais on met la validation groupée directement dans le graphe pour ne pas oublier les `groups`.

In [65]:
import skrub

data = skrub.var("visits", visits)
groups_op = data["patient_id"]

X_op = data.drop("target", axis=1).skb.mark_as_X(
    cv=GroupKFold(n_splits=5),
    split_kwargs={"groups": groups_op},
)
y_op = data["target"].skb.mark_as_y()

pred = X_op.skb.apply(skrub.TableVectorizer()).skb.apply(
    HistGradientBoostingRegressor(random_state=0),
    y=y_op,
)

report = evaluate(pred, data={"visits": visits})
report.metrics.summarize()

Output()

Output()

skrublearner_mean  skrublearner_std
metric                                           
score              7.847547e-01      3.798152e-03
r2                 7.847547e-01      3.798152e-03
rmse               7.650678e+00      1.423599e-01
mae                6.027511e+00      9.723625e-02
mape               1.104508e+14      3.522530e+13
fit_time           1.419432e+00      5.280395e-02
predict_time       1.075518e-01      2.635906e-02
Use .frame() to control the format of the output.

Attention : ici `X_op` contient encore `Index` et `patient_id` puisqu'on a seulement retiré `target`. Le modèle peut donc les utiliser, ce qui explique sans doute une différence avec l'étape 5. On les retire dans la version ci-dessous.

In [66]:
X_op = data.drop(["target", "Index", "patient_id"], axis=1, errors="ignore").skb.mark_as_X(
    cv=GroupKFold(n_splits=5),
    split_kwargs={"groups": groups_op},
)
y_op = data["target"].skb.mark_as_y()

pred = X_op.skb.apply(skrub.TableVectorizer()).skb.apply(
    HistGradientBoostingRegressor(random_state=0),
    y=y_op,
)

report = evaluate(pred, data={"visits": visits})
report.metrics.summarize()

Output()

Output()

skrublearner_mean  skrublearner_std
metric                                           
score              7.919485e-01      4.045155e-03
r2                 7.919485e-01      4.045155e-03
rmse               7.521647e+00      1.402389e-01
mae                5.925923e+00      1.000548e-01
mape               9.855692e+13      3.401739e+13
fit_time           1.019374e+00      4.214805e-02
predict_time       5.642631e-02      6.256886e-03
Use .frame() to control the format of the output.

In [67]:
project.put("06_dataops", report)

Output()

Consult your report at 
]8;id=5575189;https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/cross-validations/89867\https://skore.probabl.ai/bob-le-bricoleur/bobathon-esilv/cross-validations/89867]8;;\

## 7. Modèle final et soumission

On reprend le pipeline skrub de l'étape 5, on l'entraîne sur tout le train et on prédit le test.

In [68]:
final = clone(model).fit(X_full, y)

submission = X_test[["Index"]].copy()
submission["target"] = final.predict(X_test.drop(columns=["Index", "patient_id"], errors="ignore"))
submission.to_csv(submissions_dir / "05_tabular_pipeline.csv", index=False)
submission.head()

,Index,target
0,66,19.706307
1,67,29.851750
2,68,30.705663
3,69,35.543654
4,70,31.896403


In [69]:
# version DataOps avec un learner
learner = pred.skb.make_learner()
learner.fit({"visits": visits})
pred_test = learner.predict({"visits": X_test})

submission = X_test[["Index"]].copy()
submission["target"] = pred_test
submission.to_csv(submissions_dir / "06_dataops.csv", index=False)